In [1]:
import pandas as pd
import os

def load_level_by_id(path: str, level_id: str) -> str:
    """
    path: path to the .txt level file
    level_id: stringified triple, e.g. "000", "014"

    Returns:
        Level grid as a single string (with newlines)
    """
    target = int(level_id)  # "000" -> 0
    current_id = None
    collecting = False
    level_lines = []

    with open(path, "r") as f:
        for line in f:
            line = line.rstrip("\n")

            # Level header
            if line.startswith(";"):
                # Stop if we were collecting and hit next level
                if collecting:
                    break

                # Parse level number
                try:
                    current_id = int(line[1:].strip())
                except ValueError:
                    current_id = None

                collecting = (current_id == target)
                continue

            # Collect level lines
            if collecting:
                level_lines.append(line)

    if not level_lines:
        raise ValueError(f"Level {level_id} not found in file")

    return "\n".join(level_lines)

def fill_name(string, length=3):
    while len(string) < length:
        string = "0" + string
    return string

from typing import Set, Tuple

def parse_sokoban_level(level_str: str):
    """
    Parses a 10x10 Sokoban level string.

    Returns:
        walls  : Set[(r, c)]
        boxes  : Set[(r, c)]
        goals  : Set[(r, c)]
        player : (r, c)
    """
    walls: Set[Tuple[int, int]] = set()
    boxes: Set[Tuple[int, int]] = set()
    goals: Set[Tuple[int, int]] = set()
    player = None

    rows = level_str.split("\n")

    for r, row in enumerate(rows):
        for c, ch in enumerate(row):
            if ch == "#":
                walls.add((r, c))

            elif ch == "$":
                boxes.add((r, c))

            elif ch == ".":
                goals.add((r, c))

            elif ch == "@":
                player = (r, c)

            elif ch == "*":          # box on goal
                boxes.add((r, c))
                goals.add((r, c))

            elif ch == "+":          # player on goal
                player = (r, c)
                goals.add((r, c))

    if player is None:
        raise ValueError("No player found in level")

    return {
        "walls": walls,
        "boxes": boxes,
        "goals": goals,
        "player": player
    }   

def play(state, actions_str):
    action_map = [(-1,0),(0,1),(1,0),(0,-1)]
    states = [state]
    for action in actions_str:
        walls, boxes, goals, player = state["walls"], state["boxes"], state["goals"], state["player"]
        if tuple(sorted(boxes)) == tuple(sorted(goals)):
            print("solved")
            break
        dy, dx = action_map[int(action)]
        new_pos_player = (player[0]+dy, player[1]+dx)
        if new_pos_player in walls:
            continue
        if new_pos_player in boxes:
            new_pos_box = (new_pos_player[0]+dy, new_pos_player[1]+dx)
            if new_pos_box in boxes:
                continue
            boxes.remove(new_pos_player)
            boxes.add(new_pos_box)

        player = new_pos_player
        state = {"walls": walls, "boxes": boxes, "goals": goals, "player": player}
        states.append(state)

    return states

In [2]:
from torch.utils.data import Dataset
import torch
from tqdm import tqdm

def symbolic_state_to_tensor(state, grid_shape_x, grid_shape_y, channels):
    num_channels = len(channels)
    tensor = torch.zeros((grid_shape_x, grid_shape_y, num_channels), dtype=torch.float32)

    for c, key in enumerate(channels):
        values = state[key] if key != "player" else {state[key]}
        idx = torch.tensor(list(values), dtype=torch.long)  # shape [N, 2]
        tensor[idx[:, 0], idx[:, 1], c] = 1.0

    return tensor

class SokobanDataset(Dataset):
    def __init__(self, config):
        self.config = config
        self.data = self.load_data(config)

    def load_data(self, config):
        
        difficulty = config["difficulty"]
        subset_name = config["subset_name"]

        grid_shape_x = config["grid_shape_x"]
        grid_shape_y = config["grid_shape_y"]

        max_num_levels = config["max_num_levels"]

        channels = ['boxes', 'goals', 'player', 'walls']
        num_channels = len(channels)

        df = pd.read_csv(f"../../boxoban-astar-solutions/{difficulty}_{subset_name}.csv")
        filtered = df[
            (df["Steps"] != "INCORRECT_SOLUTION_FOUND") & (df["Actions"] != "SEARCH_STATE_FAILED")
        ]

        folder_names = filtered["File"].unique().tolist()

        data = []
        tqdm_folder_names = tqdm(folder_names, desc="Loading dataset")
        count_levels = 0
        for folder_name in tqdm_folder_names:
            folder_name = str(folder_name)
            folder_name = fill_name(folder_name)    

            level_ids = filtered[filtered["File"] == int(folder_name)]["Level"].unique().tolist()
            tqdm_level_ids = tqdm(level_ids, desc=f"Processing levels in file {folder_name}", leave=False)
            for level_id in tqdm_level_ids:
                if count_levels >= max_num_levels:
                    break
                level_id =  str(level_id)

                row_ix = filtered[(filtered["File"] == int(folder_name)) & (filtered["Level"] == int(level_id))].index[0]

                level_id = fill_name(level_id)

                level = load_level_by_id(f"../../boxoban-levels/{difficulty}/{subset_name}/{folder_name}.txt", level_id)
                actions_str = filtered.loc[row_ix]["Actions"]

                state_0 = parse_sokoban_level(level)
                states = play(state_0, actions_str)
                
                states_tensor = [
                    symbolic_state_to_tensor(state, grid_shape_x, grid_shape_y, channels)
                    for state in states[:-1]
                ]
                states_tensor = torch.stack(states_tensor, dim=0)  # shape [T, H, W, C]

                actions_id = [int(a) for a in actions_str]  # shape [T-1]
                actions_id = torch.tensor(actions_id, dtype=torch.long)

                datum = {
                    "states_tensor": states_tensor,  # shape [T-1, H, W, C]
                    "actions_id": actions_id,        # shape [T-1]
                }
                data.append(datum)
                count_levels += 1
                
        return data


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [3]:
config_dataset = {
    "difficulty": "medium",
    "subset_name": "valid",
    "grid_shape_x": 10,
    "grid_shape_y": 10,
    "max_num_levels": 16,
}

dataset = SokobanDataset(config_dataset)

Loading dataset: 100%|██████████| 50/50 [00:00<00:00, 95.64it/s] 


In [4]:
batch_size = 8

def collate_fn(batch):
    batch_states = [item["states_tensor"] for item in batch]
    batch_actions = [item["actions_id"] for item in batch]

    batch_states_padded = torch.nn.utils.rnn.pad_sequence(batch_states, batch_first=True, padding_value=0.0)
    batch_actions_padded = torch.nn.utils.rnn.pad_sequence(batch_actions, batch_first=True, padding_value=-100)

    return {
        "states_tensors": batch_states_padded,
        "actions_ids": batch_actions_padded
    }

from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
batch = next(iter(dataloader))

batch["states_tensors"].shape, batch["actions_ids"].shape   

(torch.Size([8, 65, 10, 10, 4]), torch.Size([8, 65]))

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VisualEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.in_channels = config["in_channels"]
        latent_dim = config["latent_dim"]
        channels = config["channels"]
        kernel_size = config["kernel_size"]
        padding = config["padding"]
        stride = config["stride"]
        grid_shape_x = config["grid_shape_x"]
        grid_shape_y = config["grid_shape_y"]
        self.input_hw = (grid_shape_x, grid_shape_y)

        layers = []
        prev_c = self.in_channels

        for i, out_c in enumerate(channels):
            layers.append(
                nn.Conv2d(
                    prev_c,
                    out_c,
                    kernel_size=kernel_size,
                    padding=padding,
                    stride=stride if i == len(channels) - 1 else 1,
                )
            )
            layers.append(nn.ReLU())
            prev_c = out_c

        self.cnn = nn.Sequential(*layers)

        self._cnn_out_dim = self._infer_cnn_out_dim()

        self.fc = nn.Linear(self._cnn_out_dim, latent_dim)

    def _infer_cnn_out_dim(self):
        with torch.no_grad():
            dummy = torch.zeros(1, self.in_channels, *self.input_hw)
            out = self.cnn(dummy)
            return out.numel()

    def forward(self, x):
        B, T, W, H, C = x.shape

        # (B, T, C, W, H) → (B*T, C, W, H)
        x = x.permute(0, 1, 4, 2, 3).reshape(B * T, C, W, H)

        x = self.cnn(x)
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)

        return x.view(B, T, -1)

config_visual_encoder = {
    "in_channels": 4,
    "latent_dim": 32,
    "grid_shape_x": 10,
    "grid_shape_y": 10,
    "channels": [16, 32, 32],   # out channels for each conv layer
    "kernel_size": 3,
    "padding": 1,
    "stride": 2,                # applied only to last conv
}
model = VisualEncoder(config_visual_encoder)
model

VisualEncoder(
  (cnn): Sequential(
    (0): Conv2d(4, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
  )
  (fc): Linear(in_features=800, out_features=32, bias=True)
)

In [31]:
states_tensors = batch["states_tensors"]  # shape [B, T, H, W, C]
latent_states = model(states_tensors)
latent_states.shape

torch.Size([8, 65, 32])

In [ ]:
class ActionDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.model_name = config["model_name"]
        hidden_size = config["hidden_size"]
        vocab_size = config["vocab_size"]

        args = config["args"]

        if self.model_name == "qwen2":
            from transformers import Qwen2Model, Qwen2Config
            args_model = {
                "num_hidden_layers": args["num_layers"],
                "hidden_size": args["hidden_size"],
                "num_attention_heads": args["num_attention_heads"],
                "num_key_value_heads": args["num_attention_heads"],
                "intermediate_size": args["hidden_size"] * 4,
            }
            config_model =  Qwen2Config(**args_model)
            self.backbone = Qwen2Model(config_model)

        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        latent_states = x["latent_states"]
        if self.model_name == "qwen2":
            outputs = self.backbone(inputs_embeds=latent_states)
            hidden_states = outputs.last_hidden_state
            logits = self.lm_head(hidden_states[:, :, :])

            return {
                "logits": logits,
            }
        else:
            raise ValueError(f"Unknown model name: {self.model_name}")

In [29]:
config_action_decoder = {
    "model_name": "qwen2",
    "args": {
        "num_layers": 2,
        "hidden_size": 32,
        "num_attention_heads": 1,
    },
    "hidden_size": 32,
    "vocab_size": 4,
}

action_decoder = ActionDecoder(config_action_decoder)
action_decoder.eval()

ActionDecoder(
  (backbone): Qwen2Model(
    (embed_tokens): Embedding(151936, 32)
    (layers): ModuleList(
      (0-1): 2 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=32, out_features=32, bias=True)
          (k_proj): Linear(in_features=32, out_features=32, bias=True)
          (v_proj): Linear(in_features=32, out_features=32, bias=True)
          (o_proj): Linear(in_features=32, out_features=32, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=32, out_features=128, bias=False)
          (up_proj): Linear(in_features=32, out_features=128, bias=False)
          (down_proj): Linear(in_features=128, out_features=32, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((32,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((32,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((32,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbedding()
  

In [34]:
latent_states
action_decoder({"latent_states": latent_states})

{'logits': tensor([[[-6.5633e-01, -2.1920e-01, -4.3235e-01,  5.8931e-01],
          [-6.7404e-01, -2.6264e-01, -4.5542e-01,  6.3725e-01],
          [-6.8637e-01, -1.5522e-01, -4.6614e-01,  7.1223e-01],
          ...,
          [-7.4412e-01,  1.1408e-01, -6.8336e-01,  7.3063e-01],
          [-7.4639e-01,  1.1535e-01, -6.8289e-01,  7.3039e-01],
          [-7.4861e-01,  1.1658e-01, -6.8243e-01,  7.3015e-01]],
 
         [[-7.5819e-01, -4.0487e-02, -5.9608e-01,  7.6916e-01],
          [-7.3146e-01, -7.1593e-02, -5.4450e-01,  7.7370e-01],
          [-7.2454e-01, -5.5371e-05, -6.1152e-01,  7.8582e-01],
          ...,
          [-8.0380e-01,  1.6296e-01, -7.0517e-01,  8.0385e-01],
          [-8.0575e-01,  1.6303e-01, -7.0361e-01,  8.0281e-01],
          [-8.0765e-01,  1.6311e-01, -7.0210e-01,  8.0180e-01]],
 
         [[-7.7688e-01, -8.3943e-02, -3.6991e-01,  8.3118e-01],
          [-7.0782e-01, -1.7054e-01, -4.7910e-01,  7.5888e-01],
          [-7.0818e-01, -9.0106e-02, -4.3743e-01,  7.5154e

In [38]:
class Thinker(nn.Module):
    def __init__(self, config):
        super().__init__()

        config_visual_encoder = config["config_visual_encoder"]
        self.visual_encoder = VisualEncoder(config_visual_encoder)

        config_action_decoder = config["config_action_decoder"]
        self.action_decoder = ActionDecoder(config_action_decoder)
    
    def forward(self, batch):

        states_tensors = batch["states_tensors"]  # shape [B, T, H, W, C]
        latent_states = self.visual_encoder(states_tensors)  # shape [B, T, D]
        x = {
            "latent_states": latent_states,
        }
        decoder_output = self.action_decoder(x)  # shape [B, T, num_actions]

        return {
            "decoder_output": decoder_output
        }

In [39]:
config_thinker = {
    "config_visual_encoder": config_visual_encoder,
    "config_action_decoder": config_action_decoder,
}

thinker = Thinker(config_thinker)

In [40]:
out = thinker(batch)
out

{'decoder_output': {'logits': tensor([[[ 0.3629,  1.2335,  0.4039, -0.0122],
           [ 0.3515,  1.2109,  0.4089,  0.0416],
           [ 0.3151,  1.2389,  0.3544, -0.0186],
           ...,
           [ 0.2430,  0.7393, -0.0748,  0.0123],
           [ 0.2398,  0.7353, -0.0765,  0.0122],
           [ 0.2367,  0.7315, -0.0782,  0.0121]],
  
          [[-0.1491,  0.6904,  0.1126,  0.3407],
           [-0.1070,  0.7304,  0.0407,  0.3906],
           [-0.1492,  0.7672,  0.0748,  0.3802],
           ...,
           [ 0.0153,  0.4459, -0.1015,  0.0223],
           [ 0.0158,  0.4458, -0.1029,  0.0217],
           [ 0.0162,  0.4457, -0.1043,  0.0212]],
  
          [[ 0.0024,  0.5986, -0.0451,  0.0713],
           [ 0.0092,  0.6215, -0.0196,  0.0185],
           [-0.0340,  0.5996,  0.0170,  0.0416],
           ...,
           [ 0.0735,  0.4975, -0.1733,  0.0124],
           [ 0.0729,  0.4967, -0.1739,  0.0122],
           [ 0.0724,  0.4959, -0.1744,  0.0120]],
  
          ...,
  
          [[